In [ ]:
!pip install -q datasets transformers evaluate scikit-learn accelerate


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline
)
from datasets import load_dataset, DatasetDict
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import login
print("Đã import xong")

Đã import xong


In [ ]:
from huggingface_hub import login
login()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

dataset = load_dataset("visolex/ViSFD")
print(dataset)

README.md:   0%|          | 0.00/4.00k [00:00<?, ?B/s]

ViSFD.csv:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11122 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label'],
        num_rows: 11122
    })
})


In [ ]:
print(dataset['train'][0])


{'dataset': 'UIT-ViSFD', 'type': 'train', 'index': 0, 'comment': 'Mới mua máy này Tại thegioididong thốt nốt cảm thấy ok bin trâu chụp ảnh đẹp loa nghe to bắt wf khỏe sóng ổn định, giá thành vừa với túi tiền, nhân viên tư vấn nhiệt tình', 'n_star': 5, 'date_time': '2 tuần trước', 'label': '{CAMERA#Positive};{FEATURES#Positive};{BATTERY#Positive};{PRICE#Positive};{GENERAL#Positive};{SER&ACC#Positive};'}


In [ ]:
print(dataset['train'].column_names)

['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label']


In [ ]:
import json
sample = dataset['train'][0]
print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "dataset": "UIT-ViSFD",
  "type": "train",
  "index": 0,
  "comment": "Mới mua máy này Tại thegioididong thốt nốt cảm thấy ok bin trâu chụp ảnh đẹp loa nghe to bắt wf khỏe sóng ổn định, giá thành vừa với túi tiền, nhân viên tư vấn nhiệt tình",
  "n_star": 5,
  "date_time": "2 tuần trước",
  "label": "{CAMERA#Positive};{FEATURES#Positive};{BATTERY#Positive};{PRICE#Positive};{GENERAL#Positive};{SER&ACC#Positive};"
}


In [ ]:
# Đếm số lượng mẫu theo n_star (số sao)
from collections import Counter

stars = [item['n_star'] for item in dataset['train']]
counter = Counter(stars)
print("Phân bố theo số sao:")
for star, count in sorted(counter.items()):
    print(f"  {star} sao: {count} mẫu")

Phân bố theo số sao:
  1 sao: 1831 mẫu
  2 sao: 829 mẫu
  3 sao: 1426 mẫu
  4 sao: 1851 mẫu
  5 sao: 5185 mẫu


In [ ]:
for i in range(5):
    item = dataset['train'][i]
    print(f"--- Mẫu {i} ---")
    print(f"Comment: {item['comment']}")
    print(f"Số sao: {item['n_star']}")
    print()

--- Mẫu 0 ---
Comment: Mới mua máy này Tại thegioididong thốt nốt cảm thấy ok bin trâu chụp ảnh đẹp loa nghe to bắt wf khỏe sóng ổn định, giá thành vừa với túi tiền, nhân viên tư vấn nhiệt tình
Số sao: 5

--- Mẫu 1 ---
Comment: Pin kém còn lại miễn chê mua 8/3/2019 tình trạng pin còn 88% có ai giống tôi không
Số sao: 5

--- Mẫu 2 ---
Comment: Sao lúc gọi điện thoại màn hình bị chấm nhỏ nháy gần camera trước vậy(lúc có lúc không)
Số sao: 3

--- Mẫu 3 ---
Comment: Mọi người cập nhật phần mềm lại , nó sẽ bớt tốn pin, mình đã thử rồi, mọi thứ cũng ok, nhưng vân tay ko nhạy
Số sao: 3

--- Mẫu 4 ---
Comment: Mới mua Sài được 1 tháng thấy pin rất trâu, Sài bao mượt . Nhưng có 1 lỗi nhỏ là mình nghe nhạc bằng tai nghe nghe hơi lâu ko biết sao nó ko nghe được nữa.. mất rút tai nghe ra cắm vào lại thì nó mới nghe được...
Số sao: 5



In [ ]:
def map_star_to_sentiment(n_star):
    if n_star <= 2:
        return 'NEGATIVE'
    elif n_star == 3:
        return 'NEUTRAL'
    else:
        return 'POSITIVE'

# Kiểm tra
for star in [1, 2, 3, 4, 5]:
    print(f"{star} sao → {map_star_to_sentiment(star)}")

1 sao → NEGATIVE
2 sao → NEGATIVE
3 sao → NEUTRAL
4 sao → POSITIVE
5 sao → POSITIVE


In [ ]:
def add_sentiment_label(example):
    example['sentiment'] = map_star_to_sentiment(example['n_star'])
    return example

dataset = dataset.map(add_sentiment_label)

# Kiểm tra kết quả
print(dataset['train'][0])

Map:   0%|          | 0/11122 [00:00<?, ? examples/s]

{'dataset': 'UIT-ViSFD', 'type': 'train', 'index': 0, 'comment': 'Mới mua máy này Tại thegioididong thốt nốt cảm thấy ok bin trâu chụp ảnh đẹp loa nghe to bắt wf khỏe sóng ổn định, giá thành vừa với túi tiền, nhân viên tư vấn nhiệt tình', 'n_star': 5, 'date_time': '2 tuần trước', 'label': '{CAMERA#Positive};{FEATURES#Positive};{BATTERY#Positive};{PRICE#Positive};{GENERAL#Positive};{SER&ACC#Positive};', 'sentiment': 'POSITIVE'}


In [ ]:
from collections import Counter

sentiments = [item['sentiment'] for item in dataset['train']]
counter = Counter(sentiments)
print("Phân bố nhãn cảm xúc (tập train):")
for sentiment, count in counter.items():
    print(f"  {sentiment}: {count} mẫu")

Phân bố nhãn cảm xúc (tập train):
  POSITIVE: 7036 mẫu
  NEUTRAL: 1426 mẫu
  NEGATIVE: 2660 mẫu


In [ ]:
import random
from datasets import Dataset

# Đặt seed để kết quả reproducible
random.seed(42)

# Lấy index của từng nhãn
pos_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'POSITIVE']
neg_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'NEGATIVE']
neu_indices = [i for i, item in enumerate(dataset['train']) if item['sentiment'] == 'NEUTRAL']

print(f"POSITIVE: {len(pos_indices)} mẫu")
print(f"NEGATIVE: {len(neg_indices)} mẫu")
print(f"NEUTRAL:  {len(neu_indices)} mẫu")

# Số mẫu chung = số mẫu ít nhất
n_samples = min(len(pos_indices), len(neg_indices), len(neu_indices))
print(f"\nSố mẫu chung sau undersampling: {n_samples}")

# Chọn ngẫu nhiên
pos_sampled = random.sample(pos_indices, n_samples)
neg_sampled = random.sample(neg_indices, n_samples)
neu_sampled = neu_indices  # NEUTRAL giữ nguyên vì đã ít nhất

# Gộp và xáo trộn
all_indices = pos_sampled + neg_sampled + neu_sampled
random.shuffle(all_indices)

# Tạo tập train mới
train_balanced = dataset['train'].select(all_indices)

print(f"\nTập train sau cân bằng: {len(train_balanced)} mẫu")

POSITIVE: 7036 mẫu
NEGATIVE: 2660 mẫu
NEUTRAL:  1426 mẫu

Số mẫu chung sau undersampling: 1426

Tập train sau cân bằng: 4278 mẫu


In [ ]:
from collections import Counter

sentiments = [item['sentiment'] for item in train_balanced]
counter = Counter(sentiments)
print("Phân bố nhãn sau undersampling:")
for sentiment, count in counter.items():
    print(f"  {sentiment}: {count} mẫu")

Phân bố nhãn sau undersampling:
  NEUTRAL: 1426 mẫu
  POSITIVE: 1426 mẫu
  NEGATIVE: 1426 mẫu


In [ ]:
# Lưu tập train đã cân bằng vào dataset
dataset['train'] = train_balanced

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment'],
        num_rows: 4278
    })
})


In [ ]:
print(dataset)
print()
print("Các tập có sẵn:", list(dataset.keys()))

DatasetDict({
    train: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment'],
        num_rows: 4278
    })
})

Các tập có sẵn: ['train']


In [ ]:
from datasets import DatasetDict

# Bước 1: Chia tập train_balanced thành 70% train + 30% tạm
train_testvalid = train_balanced.train_test_split(test_size=0.3, seed=42)

# Bước 2: Chia tập tạm (30%) thành 15% validation + 15% test
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Bước 3: Gộp thành DatasetDict
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment'],
        num_rows: 2994
    })
    validation: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment'],
        num_rows: 642
    })
    test: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment'],
        num_rows: 642
    })
})


In [ ]:
from collections import Counter

for split in ['train', 'validation', 'test']:
    sentiments = [item['sentiment'] for item in dataset[split]]
    counter = Counter(sentiments)
    print(f"\nPhân bố nhãn tập {split.upper()}:")
    for sentiment in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
        print(f"  {sentiment}: {counter.get(sentiment, 0)} mẫu")


Phân bố nhãn tập TRAIN:
  POSITIVE: 990 mẫu
  NEGATIVE: 1011 mẫu
  NEUTRAL: 993 mẫu

Phân bố nhãn tập VALIDATION:
  POSITIVE: 217 mẫu
  NEGATIVE: 199 mẫu
  NEUTRAL: 226 mẫu

Phân bố nhãn tập TEST:
  POSITIVE: 219 mẫu
  NEGATIVE: 216 mẫu
  NEUTRAL: 207 mẫu


In [ ]:
# Lưu dataset vào thư mục tạm để dùng cho bước sau
dataset.save_to_disk('/content/uit-visfd-split')
print("Đã lưu dataset vào /content/uit-visfd-split")

Saving the dataset (0/1 shards):   0%|          | 0/2994 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/642 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/642 [00:00<?, ? examples/s]

Đã lưu dataset vào /content/uit-visfd-split


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Đã load tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Đã load tokenizer: vinai/phobert-base
Vocab size: 64000


In [ ]:
sample_text = "Sản phẩm này rất tốt, pin trâu, màn hình đẹp"
tokens = tokenizer(sample_text, return_tensors="pt")
print(tokens)
print("\nSố token:", len(tokens['input_ids'][0]))

{'input_ids': tensor([[    0, 13834,  3884,    23,    59, 23716,  1187,     4,  2179,  2127,
         10440,  1881,     4,  1449,   819,   258,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Số token: 17


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['comment'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/2994 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment', 'input_ids', 'attention_mask'],
        num_rows: 2994
    })
    validation: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment', 'input_ids', 'attention_mask'],
        num_rows: 642
    })
    test: Dataset({
        features: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'label', 'sentiment', 'input_ids', 'attention_mask'],
        num_rows: 642
    })
})


In [ ]:
# Mapping nhãn chữ → số
label_to_id = {
    'NEGATIVE': 0,
    'NEUTRAL': 1,
    'POSITIVE': 2
}

id_to_label = {v: k for k, v in label_to_id.items()}

print("Mapping nhãn:")
for label, id in label_to_id.items():
    print(f"  {label} → {id}")

Mapping nhãn:
  NEGATIVE → 0
  NEUTRAL → 1
  POSITIVE → 2


In [ ]:
def encode_labels(example):
    example['labels'] = label_to_id[example['sentiment']]
    return example

tokenized_dataset = tokenized_dataset.map(encode_labels)

# Kiểm tra
print("Mẫu đầu tiên sau khi encode:")
print("Comment:", tokenized_dataset['train'][0]['comment'][:80], "...")
print("Sentiment gốc:", tokenized_dataset['train'][0]['sentiment'])
print("Labels số:", tokenized_dataset['train'][0]['labels'])

Map:   0%|          | 0/2994 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Mẫu đầu tiên sau khi encode:
Comment: Rất vừa ý, đẹp và sang, chắc mình mua nhầm máy ngon nên giờ chưa thấy gì bất thư ...
Sentiment gốc: POSITIVE
Labels số: 2


In [ ]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id
)

# Đưa model lên GPU
model = model.to("cuda")

print("Đã load model:", MODEL_NAME)
print("Số nhãn:", model.config.num_labels)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã load model: vinai/phobert-base
Số nhãn: 3


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
from transformers import TrainingArguments
print("Đã import TrainingArguments")

Đã import TrainingArguments


In [ ]:
training_args = TrainingArguments(
    output_dir="/content/phobert-temp",         # ← Colab local, không phải Drive
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=200,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no",                         # ← Không lưu checkpoint
    load_best_model_at_end=False,
    report_to="none",
    seed=42
)

print("Output dir:", training_args.output_dir)


Output dir: /content/phobert-temp


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)
print("Đã khởi tạo trainer")

Đã khởi tạo trainer


In [ ]:
# Xóa cột 'label' gốc (aspect-based, không dùng nữa)
# Đổi tên cột 'labels' thành 'label' để Trainer nhận diện
def fix_label_column(example):
    example['label'] = example['labels']  # copy giá trị số sang label
    return example

# Áp dụng cho cả 3 tập
tokenized_dataset = tokenized_dataset.map(fix_label_column)

# Xóa cột 'labels' cũ (không cần nữa)
tokenized_dataset = tokenized_dataset.remove_columns(['labels'])

# Kiểm tra
print("Features sau khi sửa:")
print(tokenized_dataset['train'].features)
print()
print("Mẫu đầu tiên:")
print("Comment:", tokenized_dataset['train'][0]['comment'][:60], "...")
print("Label:", tokenized_dataset['train'][0]['label'])
print("Sentiment gốc:", tokenized_dataset['train'][0]['sentiment'])

Map:   0%|          | 0/2994 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Features sau khi sửa:
{'dataset': Value('string'), 'type': Value('string'), 'index': Value('int64'), 'comment': Value('string'), 'n_star': Value('int64'), 'date_time': Value('string'), 'label': Value('int64'), 'sentiment': Value('string'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

Mẫu đầu tiên:
Comment: Rất vừa ý, đẹp và sang, chắc mình mua nhầm máy ngon nên giờ  ...
Label: 2
Sentiment gốc: POSITIVE


In [ ]:
# Xóa các cột không cần thiết cho training
columns_to_keep = ['input_ids', 'attention_mask', 'label']
columns_to_remove = [col for col in tokenized_dataset['train'].column_names
                     if col not in columns_to_keep]

print("Cột sẽ xóa:", columns_to_remove)

tokenized_dataset = tokenized_dataset.remove_columns(columns_to_remove)

# Kiểm tra
print("\nFeatures sau khi xóa:")
print(tokenized_dataset['train'].features)

Cột sẽ xóa: ['dataset', 'type', 'index', 'comment', 'n_star', 'date_time', 'sentiment']

Features sau khi xóa:
{'label': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

print("Đã khởi tạo lại Trainer")

Đã khởi tạo lại Trainer


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.049835,0.849590,0.609034,0.607590,0.609034,0.594366
2,0.710741,0.739059,0.663551,0.670547,0.663551,0.666256
3,0.678765,0.729840,0.658879,0.668021,0.658879,0.662341


TrainOutput(global_step=564, training_loss=0.8214629744807034, metrics={'train_runtime': 208.6864, 'train_samples_per_second': 43.041, 'train_steps_per_second': 2.703, 'total_flos': 590821179508224.0, 'train_loss': 0.8214629744807034, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(tokenized_dataset['test'])
print("Kết quả trên tập test:")
for key, value in test_results.items():
    if key.startswith('eval_'):
        if isinstance(value, float):
            print(f"  {key[5:]}: {value:.4f}")
        else:
            print(f"  {key[5:]}: {value}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.678765,0.734057,3,0.671340,0.680519,0.671340,0.674709


Kết quả trên tập test:
  loss: 0.7341
  accuracy: 0.6713
  precision: 0.6805
  recall: 0.6713
  f1: 0.6747


In [ ]:
trainer.save_model("./phobert-finetuned-final")
tokenizer.save_pretrained("./phobert-finetuned-final")
print("Đã lưu model vào ./phobert-finetuned-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu model vào ./phobert-finetuned-final


In [ ]:
from transformers import pipeline

# Load model đã fine-tune
classifier = pipeline(
    "text-classification",
    model="./phobert-finetuned-final",
    tokenizer="./phobert-finetuned-final",
    device=0
)

# Test 3 câu
test_reviews = [
    "Sản phẩm rất tốt, pin trâu, màn hình đẹp, giao hàng nhanh",
    "Hàng kém chất lượng, giao chậm, thất vọng",
    "Bình thường, không có gì đặc biệt"
]

for review in test_reviews:
    result = classifier(review)[0]
    print(f"Review: {review}")
    print(f"  → {result['label']} (score: {result['score']:.4f})")
    print()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Review: Sản phẩm rất tốt, pin trâu, màn hình đẹp, giao hàng nhanh
  → POSITIVE (score: 0.9324)

Review: Hàng kém chất lượng, giao chậm, thất vọng
  → NEGATIVE (score: 0.8103)

Review: Bình thường, không có gì đặc biệt
  → POSITIVE (score: 0.4733)



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Đổi base model
MODEL_NAME = "mr4/phobert-base-vi-sentiment-analysis"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id_to_label,
    label2id=label_to_id,
    ignore_mismatched_sizes=True
)

model = model.to("cuda")

print("Đã load model:", MODEL_NAME)
print("Số nhãn:", model.config.num_labels)

config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Đã load model: mr4/phobert-base-vi-sentiment-analysis
Số nhãn: 3


In [ ]:
from datasets import load_dataset, DatasetDict

# Load lại dataset gốc
raw_dataset = load_dataset("visolex/ViSFD")

# Mapping nhãn cảm xúc
def add_sentiment_label(example):
    n = example['n_star']
    if n <= 2:
        example['sentiment'] = 'NEGATIVE'
    elif n == 3:
        example['sentiment'] = 'NEUTRAL'
    else:
        example['sentiment'] = 'POSITIVE'
    return example

raw_dataset = raw_dataset.map(add_sentiment_label)

# Chia tập: 70% train, 15% val, 15% test (KHÔNG undersample)
train_testvalid = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

dataset_full = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

print("Số mẫu từng tập:")
print(f"  Train: {len(dataset_full['train'])}")
print(f"  Validation: {len(dataset_full['validation'])}")
print(f"  Test: {len(dataset_full['test'])}")

Map:   0%|          | 0/11122 [00:00<?, ? examples/s]

Số mẫu từng tập:
  Train: 7785
  Validation: 1668
  Test: 1669


In [ ]:
!ls /content/drive/MyDrive/InsightAI/

phobert-finetuned-v2


In [ ]:
print(training_args.output_dir)

/content/phobert-temp


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.629887,0.738529,0.671340,0.698695,0.671340,0.678799
2,0.578850,0.732527,0.669782,0.679882,0.669782,0.672220
3,0.520207,0.764115,0.661994,0.672362,0.661994,0.665925


TrainOutput(global_step=564, training_loss=0.5713972833985133, metrics={'train_runtime': 212.8938, 'train_samples_per_second': 42.19, 'train_steps_per_second': 2.649, 'total_flos': 590821179508224.0, 'train_loss': 0.5713972833985133, 'epoch': 3.0})

In [ ]:
print("Số epoch:", training_args.num_train_epochs)
print("Output dir:", training_args.output_dir)

Số epoch: 3
Output dir: /content/phobert-temp


In [ ]:
print("Model name:", model.config.name_or_path)

Model name: mr4/phobert-base-vi-sentiment-analysis


In [ ]:
from transformers import pipeline
import random

# Load model đã fine-tune lần 2
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0
)

# Lấy ngẫu nhiên 5 review từ tập test
random.seed(42)
sample_indices = random.sample(range(len(dataset_full['test'])), 5)

print("=" * 80)
print("TEST 5 REVIEW THỰC TỪ TẬP TEST")
print("=" * 80)

label_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

for i, idx in enumerate(sample_indices):
    item = dataset_full['test'][idx]
    review = item['comment']
    true_label = item['sentiment']

    # Dự đoán
    result = classifier(review[:512])[0]  # Giới hạn 512 ký tự
    pred_label = result['label']
    score = result['score']

    # So sánh
    match = "✅ ĐÚNG" if pred_label == true_label else "❌ SAI"

    print(f"\n--- Review {i+1} ---")
    print(f"Nội dung: {review[:200]}{'...' if len(review) > 200 else ''}")
    print(f"Nhãn thật:  {true_label}")
    print(f"Dự đoán:    {pred_label} (score: {score:.4f})")
    print(f"Kết quả:    {match}")

print("\n" + "=" * 80)

TEST 5 REVIEW THỰC TỪ TẬP TEST

--- Review 1 ---
Nội dung: Quá chán ,quá thất vọng. Pin nhanh hết, camera chế độ chụp đêm ,nét, làm đẹp thua xa 11pro. Tóm lại trải nghiệm 3 ngày nay thấy thật sự thất vọng. Nhân viên phục vụ nhiệt tình.
Nhãn thật:  NEGATIVE
Dự đoán:    NEGATIVE (score: 0.9858)
Kết quả:    ✅ ĐÚNG

--- Review 2 ---
Nội dung: Máy dùng mượt, chơi game không bị lag, pin ổn, màn hình to, không bị lỗi máy bao giờ
Nhãn thật:  POSITIVE
Dự đoán:    NEUTRAL (score: 0.9926)
Kết quả:    ❌ SAI

--- Review 3 ---
Nội dung: Máy dùng rất tốt khá ok  chơi được liên quân hơi bị lác . Nghe loa rõ to .Nhân viên phục vụ nhiệt tình.Mình ủng hộ thế giới Di động
Nhãn thật:  POSITIVE
Dự đoán:    NEUTRAL (score: 0.9928)
Kết quả:    ❌ SAI

--- Review 4 ---
Nội dung: điện thoại mẫu mã thiết kế đẹp,chụp hình rõ nét ,pin sử dụng ok,chính sách bảo hành tốt
Nhãn thật:  POSITIVE
Dự đoán:    NEUTRAL (score: 0.9929)
Kết quả:    ❌ SAI

--- Review 5 ---
Nội dung: Ko biết máy bắt sóng sim yếu hay như nào mà t

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="mr4/phobert-base-vi-sentiment-analysis"
)

test_reviews = [
    "Quá chán, quá thất vọng. Pin nhanh hết",
    "Máy dùng mượt, chơi game không bị lag",
    "Máy dùng rất tốt khá ok",
    "Điện thoại thiết kế đẹp, chụp hình rõ nét",
    "Máy bắt sóng sim yếu"
]

for review in test_reviews:
    result = classifier(review)[0]
    print(f"{review} → {result['label']} ({result['score']:.4f})")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Quá chán, quá thất vọng. Pin nhanh hết → Tiêu cực (0.9859)
Máy dùng mượt, chơi game không bị lag → Tích cực (0.9888)
Máy dùng rất tốt khá ok → Tích cực (0.9925)
Điện thoại thiết kế đẹp, chụp hình rõ nét → Tích cực (0.9928)
Máy bắt sóng sim yếu → Tiêu cực (0.9830)
